# 🛡️ CyberGuard AI – URL & Log Analyzer Agent

An AI-based tool designed to:
- Detect suspicious URLs (phishing)
- Analyze system logs (brute force etc.)
- Provide a Risk Level & Recommendations

---

## Agenda
1. **Setup**: Install libraries and configure API keys
2. **Tools**: Build the URL and Log analysis functions
3. **Structured Output**: Force the AI to output a standard `RiskAssessment`
4. **The Agent**: Connect Pydantic AI with our tools and structure
5. **Gradio UI**: Build a simple interactive web app for users

## Setup

In [ ]:
!pip install -q litellm pydantic-ai gradio python-dotenv

: 

In [ ]:
import os
api_key = os.getenv("GROQ_API_KEY")

print("GROQ_API_KEY:", "set ✓" if os.getenv("GROQ_API_KEY") else "MISSING ✗")

: 

---

## Part 1 — Defining the Security Tools

We need two tools for the LLM to use:
1. **`analyze_url`**: Checks for HTTP vs HTTPS, long URLs, and suspicious domains.
2. **`analyze_logs`**: Checks for multiple failed login attempts or unusual activity patterns.

In [ ]:
def analyze_url(url: str) -> str:
    """Analyze a URL for potential security risks.
    
    Returns a textual analysis of whether the URL uses HTTPS, its length, and general domain safety.
    """
    issues = []
    
    if url.startswith("http://"):
        issues.append("Insecure connection (HTTP instead of HTTPS).")
    elif not url.startswith("https://"):
        issues.append("Unknown protocol (Not HTTP or HTTPS).")
        
    if len(url) > 75:
        issues.append("Unusually long URL (Potential hiding of malicious payload).")
        
    # Simple domain check logic
    suspicious_keywords = ["login", "update", "free", "secure", "verify"]
    if any(keyword in url.lower() for keyword in suspicious_keywords):
        issues.append("Domain contains suspicious keywords commonly used in phishing.")
        
    if not issues:
        return "URL appears structurally safe. Uses HTTPS and has normal length."
    
    return "Issues found: " + " | ".join(issues)


def analyze_logs(logs: str) -> str:
    """Analyze raw system logs for attack patterns.
    
    Returns a textual analysis of potential Brute Force or unauthorized access attempts.
    """
    logs_lower = logs.lower()
    failed_attempts = logs_lower.count("failed") + logs_lower.count("error")
    
    if "brute force" in logs_lower:
        return "CRITICAL: Brute force attack explicitly mentioned in logs."
        
    if failed_attempts > 3:
        return f"HIGH RISK: Detected {failed_attempts} failed login/error attempts. Possible Brute Force."
        
    if "unauthorized" in logs_lower:
        return "MEDIUM RISK: Unauthorized access attempt detected."
        
    return "LOW RISK: Logs appear normal with no obvious attack patterns."

print("Security tools ready ✓")

---

## Part 2 — Structured Output (Pydantic)

We want the LLM to output a strict, structured Risk Assessment every time. We define the desired shape using a Pydantic `BaseModel`.

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal, List

class RiskAssessment(BaseModel):
    summary: str = Field(description="Summary of whether the inputs are safe or suspicious and the main reasons.")
    risk_level: Literal['LOW', 'MEDIUM', 'HIGH'] = Field(description="Overall calculated risk level.")
    suggestions: List[str] = Field(description="Actionable recommendations like 'Block URL', 'Enable 2FA', etc.")

print("Pydantic structure ready ✓")

---

## Part 3 — The CyberGuard Agent

Now we create the `Agent` using Pydantic AI, attaching our `RiskAssessment` as the required output.

In [ ]:
from pydantic_ai import Agent

# Using a versatile model for reasoning and tool use
cyber_agent = Agent(
    model="groq:llama-3.3-70b-versatile",
    output_type=RiskAssessment,
    system_prompt=(
        "You are CyberGuard AI, an expert cybersecurity assistant. "
        "Your job is to protect users from phishing links and system attacks. "
        "You will be given a URL and a snippet of system logs (or sometimes just one of them). "
        "Use the provided tools to analyze the URL and the logs. "
        "Combine the tool results and your own knowledge to produce a strict RiskAssessment." 
    )
)

# Attach our Python functions as tools the LLM can call naturally
cyber_agent.tool_plain(analyze_url)
cyber_agent.tool_plain(analyze_logs)

print("CyberGuard Agent is ready ✓")

In [ ]:
import asyncio

async def test_agent():
    test_url = "http://secure-login-verify.com/admin_auth"
    test_logs = """
    10:00:01 - User 'admin' login failed
    10:00:02 - User 'admin' login failed
    10:00:03 - User 'admin' login failed
    10:00:04 - User 'admin' login failed
    """

    prompt = f"Analyze this URL: {test_url}\nAnd these logs:\n{test_logs}"
    
    print("Analyzing...\n")
    result = await cyber_agent.run(prompt)
    assessment: RiskAssessment = result.output
    
    print(assessment.model_dump_json(indent=2))

# Run the test
await test_agent()

---

## Part 4 — Gradio UI

Wrap our agent in a clean Web UI so students and normal users can interact with it easily.

In [ ]:
import gradio as gr
import json

async def run_assessment(url: str, logs: str):
    # If both are empty, prompt user to enter something
    if not url.strip() and not logs.strip():
        return "Please provide either a URL or System Logs to analyze."
        
    prompt = f"Analyze this URL: {url}\nAnd these logs:\n{logs}"
    
    try:
        result = await cyber_agent.run(prompt)
        assessment = result.output
        
        # Return as pretty-printed JSON string for display
        return assessment.model_dump_json(indent=2)
    except Exception as e:
        return f"Error during analysis: {str(e)}"

# Build the Gradio interface
with gr.Blocks(title="CyberGuard AI") as demo:
    gr.Markdown("# 🛡️ CyberGuard AI – URL & Log Analyzer Agent")
    gr.Markdown("Detect phishing links and analyze system logs for brute force attacks.")
    
    with gr.Row():
        with gr.Column():
            url_input = gr.Textbox(label="Enter URL", placeholder="e.g. http://login.free-update.com")
            log_input = gr.Textbox(label="Paste System Logs", lines=8, placeholder="e.g.\nLogin Failed\nLogin Failed\n...")
            submit_btn = gr.Button("Analyze Security Risks", variant="primary")
            
        with gr.Column():
            output_json = gr.Code(label="Risk Assessment (JSON)", language="json")
            
    submit_btn.click(fn=run_assessment, inputs=[url_input, log_input], outputs=output_json)
    
    gr.Examples(
        examples=[
            ["http://secure-update-free.net/verify", "12:00 - Failed pass\n12:01 - Failed pass\n12:01 - Failed pass\n12:02 - Failed pass"],
            ["https://github.com/login", "09:00 - User admin logged in successfully"],
        ],
        inputs=[url_input, log_input]
    )

# Launch the app
demo.launch(debug=True)